In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# ==========================================
# Event Hub Connection
# ==========================================

connectionString = (
    "Endpoint=sb://eh-social-media.servicebus.windows.net/;"
    "SharedAccessKeyName=sudheer;"
    "SharedAccessKey=1IhrynvVGzNbhQLyvqMUxuT7QdTRHRh57+AEhJK7+eQ=;"
    "EntityPath=tweet-hub"
)

ehConf = {}

ehConf["eventhubs.connectionString"] = (
    sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString)
)

ehConf["eventhubs.consumerGroup"] = "$Default"

startingEventPosition = {
    "offset": "-1",
    "seqNo": -1,
    "enqueuedTime": None,
    "isInclusive": True
}

ehConf["eventhubs.startingPosition"] = json.dumps(startingEventPosition)
ehConf["eventhubs.maxEventsPerTrigger"] = "5000"

# ==========================================
# Read Stream
# ==========================================

rawDF = (
    spark.readStream
         .format("eventhubs")
         .options(**ehConf)
         .load()
)

eventDF = rawDF.selectExpr(
    "CAST(body AS STRING) AS message"
)

# ==========================================
# Schema
# ==========================================

# ==========================================
# Schema
# ==========================================

schema = StructType([
    StructField("tweet_id", StringType(), True),
    StructField("user_id", DoubleType(), True),
    StructField("tweet_text", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("timestamp_1", StringType(), True),
    StructField("likes", DoubleType(), True),
    StructField("retweets", DoubleType(), True),
    StructField("replies", DoubleType(), True),
    StructField("impressions", DoubleType(), True),
    StructField("engagement", DoubleType(), True)
])
# ==========================================
# Parse JSON
# ==========================================

parsedDF = (
    eventDF
        .select(from_json(col("message"), schema).alias("data"))
        .select("data.*")
)

# ==========================================
# Error Records
# ==========================================

errorDF = parsedDF.filter(col("tweet_id").isNull())

errorQuery = (
    errorDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/error_bronze_tweets"
        )
        .toTable("socialmedia_databricks.error.error_bronze_tweets")
)

# ==========================================
# Bronze Data
# ==========================================

bronzeDF = (
    parsedDF
    .filter(col("tweet_id").isNotNull())

    .withColumn(
        "timestamp_1",
        to_timestamp(col("timestamp_1"), "dd-MM-yyyy HH:mm")
    )

    .withColumn("bronze_load_time", current_timestamp())
    .withColumn("pipeline_name", lit("Bronze_Tweets"))
    .withColumn("source_system", lit("Azure Event Hub"))
    .withColumn("ingestion_date", current_date())

    .withWatermark("timestamp_1", "10 minutes")
)

# ==========================================
# Write Bronze Table
# ==========================================

bronzeQuery = (
    bronzeDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/bronze_tweets"
        )
        .option("mergeSchema", "true")
        .toTable("socialmedia_databricks.bronze.bronze_tweets")
)

bronzeQuery.awaitTermination()
errorQuery.awaitTermination()

print("Bronze Tweets Loaded Successfully")

Bronze Tweets Loaded Successfully


In [0]:
%sql
SELECT
    *
FROM socialmedia_databricks.bronze.bronze_tweets
LIMIT 10;

tweet_id,user_id,tweet_text,timestamp,timestamp_1,likes,retweets,replies,impressions,engagement,bronze_load_time,pipeline_name,source_system,ingestion_date
T24019,8066.0,Love this!!! 😊,api,2025-01-02T02:19:00Z,18022.0,173.0,760.0,1677.0,1374.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T32368,6805.0,Good & Bad mixed!!!,android,2025-01-03T07:15:00Z,11258.0,2134.0,380.0,1759.0,NaN,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T13032,2244.0,Good & Bad mixed!!!,api,2025-01-02T01:30:00Z,18422.0,1267.0,990.0,2367.0,NaN,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T13908,5031.0,Love this!!! 😊,ios,2025-01-22T20:46:00Z,10701.0,4507.0,153.0,606.0,1218.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T22901,6393.0,Good & Bad mixed!!!,android,2025-01-08T07:53:00Z,2372.0,3128.0,939.0,2301.0,644.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T37067,6128.0,Error###Detected,web,2025-01-12T01:25:00Z,NaN,1602.0,251.0,2460.0,1808.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T37306,9926.0,Testing!!!@@@###,android,2025-01-23T03:22:00Z,10443.0,4993.0,395.0,926.0,486.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T29585,9846.0,Love this!!! 😊,ios,2025-01-23T20:53:00Z,12627.0,892.0,923.0,2808.0,1357.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T7249,9046.0,Great product!!! #AI,api,2025-01-17T14:08:00Z,10976.0,NaN,232.0,2369.0,1856.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09
T37623,8632.0,Worst service ever :(,android,2025-01-06T09:58:00Z,5435.0,4825.0,NaN,1178.0,1540.0,2026-07-09T06:55:13.796Z,Bronze_Tweets,Azure Event Hub,2026-07-09


In [0]:
eventDF = rawDF.selectExpr("CAST(body AS STRING) AS message")

In [0]:
%sql
DROP TABLE IF EXISTS socialmedia_databricks.error.error_bronze_tweets;